# ★ 업종 설정

In [1]:
# ============================================================
# ★ 업종 설정 - 여기만 변경하면 됩니다
# ============================================================
INDUSTRY_MAP = {
    "M03": "M03_음식료품_제조업",
    "M04": "M04_섬유_가죽_신발_제조업",
    "M08": "M08_화학_의약품_고무_플라스틱_제조업",
    "M10": "M10_제1차금속산업",
    "M11": "M11_조립금속제품_제조업",
    "M12": "M12_기타기계장비_제조업",
    "M13": "M13_전자부품_컴퓨터_전기장비_제조업",
    "M15": "M15_운송장비_제조업",
    "M17": "M17_전기_가스_수도사업",
    "M18": "M18_건설업",
    "M19": "M19_도매_소매업",
    "M20": "M20_숙박_음식점업",
    "M21": "M21_운수_창고업",
    "M22": "M22_정보통신업",
    "M23": "M23_부동산_임대_사업서비스업",
    "M25": "M25_오락_문화_개인서비스업",
}

# 전체 업종 한번에 실행 (True) / 단일 업종만 실행 (False)
RUN_ALL = True

# RUN_ALL = False 일 때만 아래 단일 업종 코드 사용
SINGLE_CODE = "M19"


# Step 1. 각 파일에서 6가지 점수 데이터 불러오기 → 통합_스코어_데이터.csv

In [2]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import os

GLOBAL_MINMAX = False

def read_csv_safe(filename, **kwargs):
    df = pd.read_csv(filename, dtype={'사업자등록번호': str}, **kwargs)
    if '사업자등록번호' in df.columns:
        df['사업자등록번호'] = df['사업자등록번호'].str.zfill(10)
    return df

def minmax(s):
    mn, mx = s.min(skipna=True), s.max(skipna=True)
    if pd.isna(mn) or pd.isna(mx) or mx == mn:
        return pd.Series(np.where(s.notna(), 50.0, np.nan), index=s.index)
    return (s - mn) / (mx - mn) * 100

def minmax_by_year(df, value_col, year_col='연도'):
    return df.groupby(year_col)[value_col].transform(minmax)


run_list = list(INDUSTRY_MAP.items()) if RUN_ALL else [(SINGLE_CODE, INDUSTRY_MAP[SINGLE_CODE])]

for INDUSTRY_CODE, INDUSTRY_NAME in run_list:
    print(f'\n{"="*60}')
    print(f'  처리 중: {INDUSTRY_CODE} | {INDUSTRY_NAME}')
    print(f'{"="*60}')

    # ── 출력 폴더 생성 ──────────────────────────────────────────
    OUT_DIR  = os.path.join('22_SCORE', INDUSTRY_NAME)
    os.makedirs(OUT_DIR, exist_ok=True)
    OUT_PATH = os.path.join(OUT_DIR, '통합_스코어_데이터.csv')

    # ── 1. 생애주기 ─────────────────────────────────────────────
    # 파일명: {INDUSTRY_NAME}_lifecycle_scored_yearly_minmax.csv
    lifecycle = read_csv_safe(
        os.path.join(r'..\중간결과\20_생애주기', INDUSTRY_NAME,
                     f'{INDUSTRY_NAME}_lifecycle_scored_yearly_minmax.csv')
    )
    base = lifecycle.rename(columns={'기준연도': '연도'})[
        ['사업자등록번호', '연도', '회계년도', '생애주기_최종', '생애주기_점수', '부실라벨_ICR3년']
    ].copy()

    # ── 2. 포터 5F ──────────────────────────────────────────────
    # 파일명: 마이클포터5F.csv (고정)
    porter = read_csv_safe(
        os.path.join(r'..\중간결과\19번 마이클 포터', INDUSTRY_NAME,
                     '마이클포터5F.csv')
    )
    porter = porter[['연도', '최종점수']].copy()
    porter['porter_5F_minmax'] = minmax(porter['최종점수'])
    porter = porter[['연도', 'porter_5F_minmax']]

    # ── 3. 산업충격민감도 ────────────────────────────────────────
    # ※ 파일명 패턴 확인 필요 → 현재 추정: 산업충격민감도_OLS.csv (고정)
    ind_ols = read_csv_safe(
        os.path.join(r'..\중간결과\18번 산업별 충격민감도', INDUSTRY_NAME,
                     '산업충격민감도_OLS.csv')
    )
    ind_ols = ind_ols.rename(columns={'테스트_연도': '연도'})[['연도', 'beta_i']].copy()
    ind_ols['산업베타_minmax'] = minmax(ind_ols['beta_i'])
    ind_ols = ind_ols[['연도', '산업베타_minmax']]

    # ── 4. 기업충격민감도 ────────────────────────────────────────
    # ※ 파일명 패턴 확인 필요 → 현재 추정: 충격민감도_OLS.csv (고정)
    firm_ols = read_csv_safe(
        os.path.join(r'..\중간결과\17번 기업별 충격민감도', INDUSTRY_NAME,
                     '충격민감도_OLS.csv')
    )
    firm_ols = firm_ols.rename(columns={'테스트_연도': '연도'})[
        ['사업자등록번호', '회사명', '연도', 'beta_i']
    ].copy()
    if GLOBAL_MINMAX:
        firm_ols['기업베타_minmax'] = minmax(firm_ols['beta_i'])
    else:
        firm_ols['기업베타_minmax'] = minmax_by_year(firm_ols, 'beta_i')
    firm_ols_for_merge = firm_ols[['사업자등록번호', '연도', '기업베타_minmax']]
    firm_name_map = firm_ols[['사업자등록번호', '회사명']].drop_duplicates(subset=['사업자등록번호'])

    # ── 5. 부실확률 차이값 ───────────────────────────────────────
    # 파일명: 2015-2024_기업_부실확률_차이값.csv (고정)
    prob = read_csv_safe(
        os.path.join(r'..\중간결과\21_PD변화율', INDUSTRY_NAME,
                     '2015-2024_기업_부실확률_차이값.csv')
    )
    years = range(2015, 2025)
    long_rows = []
    for y in years:
        tmp = prob[['사업자등록번호', '회사명', f'prob_{y}', f'diff_{y-1}_{y}']].copy()
        tmp.columns = ['사업자등록번호', '회사명', 'prob', 'diff']
        tmp['연도'] = y
        long_rows.append(tmp)
    prob_long = pd.concat(long_rows, ignore_index=True)
    if GLOBAL_MINMAX:
        prob_long['부실확률_minmax']    = minmax(prob_long['prob'])
        prob_long['부실확률변화_minmax'] = minmax(prob_long['diff'])
    else:
        prob_long['부실확률_minmax']    = minmax_by_year(prob_long, 'prob')
        prob_long['부실확률변화_minmax'] = minmax_by_year(prob_long, 'diff')
    prob_for_merge = prob_long[['사업자등록번호', '회사명', '연도', '부실확률_minmax', '부실확률변화_minmax']]

    # ── 6. 병합 ─────────────────────────────────────────────────
    df = base.copy()
    df = df.merge(porter,             on='연도',                    how='left')
    df = df.merge(ind_ols,            on='연도',                    how='left')
    df = df.merge(firm_ols_for_merge, on=['사업자등록번호', '연도'], how='left')
    df = df.merge(prob_for_merge,     on=['사업자등록번호', '연도'], how='left', suffixes=('', '_prob'))
    df['회사명'] = df['회사명'].combine_first(
        df['사업자등록번호'].map(firm_name_map.set_index('사업자등록번호')['회사명'])
    )

    # ── 7. 컬럼 정리 및 저장 ────────────────────────────────────
    df = df.rename(columns={
        'porter_5F_minmax':   'Porter5F',
        '생애주기_점수':       '생애주기점수',
        '산업베타_minmax':     '산업충격민감도',
        '기업베타_minmax':     '기업충격민감도',
        '부실확률_minmax':     '부실확률',
        '부실확률변화_minmax': '부실확률변화',
    })
    score_cols = ['생애주기점수', 'Porter5F', '산업충격민감도', '기업충격민감도', '부실확률', '부실확률변화']
    final_cols = ['사업자등록번호', '회사명', '연도', '생애주기_최종', '부실라벨_ICR3년'] + score_cols
    df_final = df[final_cols].sort_values(['사업자등록번호', '연도']).reset_index(drop=True)
    df_final[score_cols] = df_final[score_cols].round(2)

    before_n = len(df_final)
    df_final = df_final.dropna(subset=['부실확률']).reset_index(drop=True)
    print(f"  '부실확률' 결측 삭제: {before_n} → {len(df_final)} ({before_n - len(df_final)}행 제거)")

    df_final.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')
    print(f'  저장 완료: {OUT_PATH}  {df_final.shape}')

print('\n✅ 전체 완료')



  처리 중: M03 | M03_음식료품_제조업


FileNotFoundError: [Errno 2] No such file or directory: '..\\중간결과\\20_생애주기\\M03_음식료품_제조업\\M03_음식료품_제조업_lifecycle_scored_yearly_minmax.csv'

# Step 2. 신용등급 병합 + 최적 가중치 그리드서치

In [ ]:
# -*- coding: utf-8 -*-
import re
import pandas as pd
import numpy as np
from scipy.stats import spearmanr

def gen_combos(total_units, n_vars):
    if n_vars == 1:
        yield (total_units,)
        return
    for i in range(total_units + 1):
        for rest in gen_combos(total_units - i, n_vars - 1):
            yield (i,) + rest

def is_long_term(rating):
    if pd.isna(rating): return False
    return not re.search(r'\d', str(rating).split('/')[0])

grade_order = ['AAA','AA+','AA','AA-','A+','A','A-',
               'BBB+','BBB','BBB-','BB+','BB','BB-',
               'B+','B','B-','CCC+','CCC','CCC-','CC','C','D']
grade_map = {g: i+1 for i, g in enumerate(grade_order)}

manual_fix = {
    'AA+STABLE':'AA+','AA+POSTIVE':'AA+','AASTABLE':'AA',
    'AA-STABLE':'AA-','A+STABLE':'A+',
}

pd_col     = '부실확률'
other_cols = ['Porter5F','생애주기점수','산업충격민감도','기업충격민감도','부실확률변화']
RECALL_MIN = 0.5

run_list = list(INDUSTRY_MAP.items()) if RUN_ALL else [(SINGLE_CODE, INDUSTRY_MAP[SINGLE_CODE])]

for INDUSTRY_CODE, INDUSTRY_NAME in run_list:
    print(f'\n{"="*60}')
    print(f'  그리드서치: {INDUSTRY_CODE} | {INDUSTRY_NAME}')
    print(f'{"="*60}')

    OUT_DIR   = os.path.join('22_SCORE', INDUSTRY_NAME)
    os.makedirs(OUT_DIR, exist_ok=True)
    score_path = os.path.join(OUT_DIR, '통합_스코어_데이터.csv')

    # 스코어 데이터
    score_df = pd.read_csv(score_path, dtype={'사업자등록번호': str})
    score_df['사업자등록번호'] = score_df['사업자등록번호'].str.zfill(10)

    # 신용등급
    credit_df = pd.read_excel(
        r'..\데이터수집\신용등급\상장사 신용등급.xlsx',
        dtype={'사업자등록번호': str}
    )
    credit_df['사업자등록번호'] = credit_df['사업자등록번호'].str.strip().str.replace('-','',regex=False).str.zfill(10)
    credit_df['연도'] = credit_df['회계년도'].astype(str).str.split('/').str[0].astype(int)
    credit_df['월']   = credit_df['회계년도'].astype(str).str.split('/').str[1].astype(int)
    credit_filtered = credit_df[credit_df['평가사구분'].isin([10])].copy()
    credit_filtered = credit_filtered[credit_filtered['신용등급'].apply(is_long_term)].copy()
    idx_latest = credit_filtered.groupby(['사업자등록번호','연도'])['월'].idxmax()
    credit_filtered = credit_filtered.loc[idx_latest].copy()
    credit_filtered['등급_base'] = credit_filtered['신용등급'].str.split('/').str[0]
    credit_filtered = credit_filtered.drop_duplicates(
        subset=['사업자등록번호','연도','월','평가사명 및 등급','신용등급']
    )
    credit_for_merge = credit_filtered[['사업자등록번호','연도','평가사명 및 등급','신용등급','평가사구분']].copy()
    credit_for_merge = credit_for_merge.rename(columns={'연도':'연도_신용등급'})

    score_df['연도_신용등급'] = score_df['연도'] + 1
    merged = score_df.merge(credit_for_merge, on=['사업자등록번호','연도_신용등급'], how='inner')

    merged['등급_정제'] = merged['신용등급'].str.split(r'[/(\s]').str[0].str.strip().str.upper()
    merged['등급_정제'] = merged['등급_정제'].replace(manual_fix)
    merged['등급_순위'] = merged['등급_정제'].map(grade_map)
    merged = merged.dropna(subset=['등급_순위']).copy()
    merged['등급_순위'] = merged['등급_순위'].astype(int)

    # 신용등급 병합 저장
    merged.to_csv(os.path.join(OUT_DIR, '통합_스코어_신용등급_병합.csv'), index=False, encoding='utf-8-sig')

    if len(merged) < 10:
        print(f'  [SKIP] 샘플 수 부족 ({len(merged)}개)')
        continue

    # 그리드서치
    y = merged['등급_순위'].values
    n_other  = len(other_cols)
    results  = []
    for pd_units in range(10, 15):   # PD 가중치 0.50~0.70 (하한 0.5 제약)
        w_pd = round(pd_units * 0.05, 2)
        remaining_total = 20 - pd_units
        extra_units     = remaining_total - n_other
        if extra_units < 0: continue
        for extra_combo in gen_combos(extra_units, n_other):
            weights_other = [round((1 + e) * 0.05, 2) for e in extra_combo]
            total_w = round(w_pd + sum(weights_other), 2)
            if abs(total_w - 1.0) > 0.01: continue
            score = merged[pd_col].values * w_pd
            for col, w in zip(other_cols, weights_other):
                score = score + merged[col].values * w
            rho, p = spearmanr(score, y)
            results.append({
                '부실확률_w': w_pd,
                **{f'{c}_w': w for c, w in zip(other_cols, weights_other)},
                '가중치합': total_w, 'spearman_rho': round(rho,4), 'p_value': round(p,4),
            })

    res_df = pd.DataFrame(results)
    res_df['abs_rho'] = res_df['spearman_rho'].abs()
    res_df_sorted = res_df.sort_values(['p_value','abs_rho'], ascending=[True,False])

    display_cols = ['부실확률_w'] + [f'{c}_w' for c in other_cols] + ['가중치합','spearman_rho','p_value']
    res_df_sorted[display_cols].to_csv(
        os.path.join(OUT_DIR, '가중치_그리드서치_결과.csv'), index=False, encoding='utf-8-sig'
    )

    best = res_df_sorted.iloc[0]
    print(f'  샘플 수: {len(merged)} | 최적 rho={best["spearman_rho"]:.4f} p={best["p_value"]:.4f}')
    print(f'  가중치: 부실확률={best["부실확률_w"]:.2f}', end='')
    for c in other_cols:
        print(f' | {c}={best[f"{c}_w"]:.2f}', end='')
    print()
    print(f'  저장 완료: {OUT_DIR}')

print('\n✅ 그리드서치 전체 완료')


# Step 3. 최적 가중치 기반 최종 스코어 산출 + 등급 부여 + 신용등급 병합

In [ ]:
import pandas as pd
import os

run_list = list(INDUSTRY_MAP.items()) if RUN_ALL else [(SINGLE_CODE, INDUSTRY_MAP[SINGLE_CODE])]

for INDUSTRY_CODE, INDUSTRY_NAME in run_list:
    OUT_DIR = os.path.join('22_SCORE', INDUSTRY_NAME)
    weights_df = pd.read_csv(os.path.join(OUT_DIR, '가중치_그리드서치_결과.csv'))
    best = weights_df.iloc[0]

    score_df = pd.read_csv(os.path.join(OUT_DIR, '통합_스코어_데이터.csv'))
    score_df['최종합산스코어'] = (
        score_df['부실확률']   * best['부실확률_w'] +
        score_df['Porter5F']  * best['Porter5F_w'] +
        score_df['생애주기점수'] * best['생애주기점수_w'] +
        score_df['산업충격민감도'] * best['산업충격민감도_w'] +
        score_df['기업충격민감도'] * best['기업충격민감도_w'] +
        score_df['부실확률변화'] * best['부실확률변화_w']
    ).round(1)

    # 등급 부여
    def assign_grade(score):
        if score >= 60: return '매우 위험'
        elif score >= 50: return '위험'
        elif score >= 40: return '중립'
        elif score >= 30: return '안정'
        else: return '매우 안정'

    score_df['등급_5단계'] = score_df['최종합산스코어'].apply(assign_grade)

    # 신용등급 병합
    rating_df = pd.read_csv(
        os.path.join(OUT_DIR, '통합_스코어_신용등급_병합.csv'),
        dtype={'사업자등록번호': str}
    )
    score_df = pd.merge(
        score_df,
        rating_df[['사업자등록번호','연도','신용등급']],
        on=['사업자등록번호','연도'], how='left'
    )

    out_path = os.path.join(OUT_DIR, '통합_스코어_데이터.csv')
    score_df.to_csv(out_path, index=False, encoding='utf-8-sig')
    print(f'[{INDUSTRY_NAME}] 저장 완료: {out_path}')

print('\n✅ 최종 스코어 산출 완료')
